<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/AlphaFold_Protein_Structure_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  AlphaFold-Based Protein Structure & Function Analysis

# Project 16 Advanced Bioinformatics

Is notebook mein hum **real AlphaFold-predicted 3D protein structures** (AlphaFold Protein Structure Database — `alphafold.ebi.ac.uk`, DeepMind/EMBL-EBI) aur **real UniProt functional annotations** use kar ke ek **structural bioinformatics pipeline** banain gay: confidence analysis, structural feature extraction, function category classification, aur interactive 3D visualization.

---

## 📋 Table of Contents

| Section | Content |
|---|---|
| 1 | Setup & Installation |
| 2 | Protein Panel — Real UniProt Accessions |
| 3 | Fetch Real AlphaFold Structures (PDB + Confidence) |
| 4 | Fetch Real UniProt Functional Annotations |
| 5 | Per-Residue Confidence (pLDDT) Analysis |
| 6 | Structural Feature Extraction |
| 7 | Exploratory Visualization |
| 8 | Function Category Classification (ML) |
| 9 | Interactive 3D Structure Viewer |
| 10 |  **Runtime Prediction** — Apna UniProt Accession Daal Kar Structure Analyze Karein |


## 1. Setup & Installation

py3Dmol real 3D molecular visualization ke liye, biopython structure parsing ke liye.


In [1]:
!pip install -q biopython py3Dmol plotly scikit-learn pandas numpy requests ipywidgets

import numpy as np
import pandas as pd
import requests
import warnings
warnings.filterwarnings("ignore")

from io import StringIO
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa

import py3Dmol

import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)
ALPHAFOLD_API = "https://alphafold.ebi.ac.uk/api/prediction"
UNIPROT_API = "https://rest.uniprot.org/uniprotkb"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 45.3 MB/s eta 0:00:00


## 2. Protein Panel — Real Human UniProt Accessions

20 well-characterized real human proteins, spanning diverse functional categories (enzymes, receptors, transport proteins, structural proteins, immune proteins).


In [2]:
protein_panel = {
    "P00734": "Prothrombin", "P07477": "Trypsin-1", "P00918": "Carbonic anhydrase 2",
    "P00441": "Superoxide dismutase [Cu-Zn]", "P9WHI7": "Catalase-peroxidase",
    "P00533": "EGFR", "P08588": "Beta-1 adrenergic receptor", "P41180": "Calcium-sensing receptor",
    "P69905": "Hemoglobin subunit alpha", "P02768": "Serum albumin", "P02753": "Retinol-binding protein 4",
    "P60709": "Actin, cytoplasmic 1", "P08670": "Vimentin", "P02452": "Collagen alpha-1(I) chain",
    "P68871": "Hemoglobin subunit beta", "P01834": "Immunoglobulin kappa constant",
    "P61769": "Beta-2-microglobulin", "P01009": "Alpha-1-antitrypsin",
    "P02787": "Serotransferrin", "P01308": "Insulin",
}
print(f"Protein panel size: {len(protein_panel)}")


Protein panel size: 20


## 3. Fetch Real AlphaFold Structures (PDB + Confidence)

In [3]:
def fetch_alphafold_structure(accession):
    """Fetch real AlphaFold predicted structure metadata + PDB file for a UniProt accession."""
    resp = requests.get(f"{ALPHAFOLD_API}/{accession}", timeout=30)
    resp.raise_for_status()
    entries = resp.json()
    if not entries:
        raise RuntimeError("no AlphaFold entry found")
    entry = entries[0]
    pdb_resp = requests.get(entry["pdbUrl"], timeout=30)
    pdb_resp.raise_for_status()
    return entry, pdb_resp.text

def parse_structure(pdb_text, accession):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(accession, StringIO(pdb_text))
    residues = [r for r in structure.get_residues() if is_aa(r)]
    plddt_scores = [r["CA"].get_bfactor() for r in residues if "CA" in r]  # AlphaFold stores pLDDT in B-factor
    ca_coords = np.array([r["CA"].get_coord() for r in residues if "CA" in r])
    return residues, np.array(plddt_scores), ca_coords

structure_records = []
pdb_cache = {}
fetch_log = []

for accession, name in protein_panel.items():
    try:
        entry, pdb_text = fetch_alphafold_structure(accession)
        residues, plddt, coords = parse_structure(pdb_text, accession)
        pdb_cache[accession] = pdb_text
        structure_records.append({
            "accession": accession, "name": name, "n_residues": len(plddt),
            "mean_pLDDT": np.mean(plddt), "pct_very_high_conf": np.mean(plddt > 90) * 100,
            "pct_low_conf": np.mean(plddt < 50) * 100, "coords": coords, "plddt": plddt,
            "source": "real"
        })
        fetch_log.append(f" {accession} ({name})")
    except Exception as e:
        fetch_log.append(f" {accession} ({name}) — {str(e)[:50]}")

print(f"Successfully fetched {len(structure_records)} / {len(protein_panel)} real AlphaFold structures")
for line in fetch_log[:10]:
    print(" ", line)


Successfully fetched 20 / 20 real AlphaFold structures
  ✅ P00734 (Prothrombin)
  ✅ P07477 (Trypsin-1)
  ✅ P00918 (Carbonic anhydrase 2)
  ✅ P00441 (Superoxide dismutase [Cu-Zn])
  ✅ P9WHI7 (Catalase-peroxidase)
  ✅ P00533 (EGFR)
  ✅ P08588 (Beta-1 adrenergic receptor)
  ✅ P41180 (Calcium-sensing receptor)
  ✅ P69905 (Hemoglobin subunit alpha)
  ✅ P02768 (Serum albumin)


In [4]:
# Fallback for any protein that failed to fetch (keeps the panel complete for downstream analysis)
rng = np.random.default_rng(42)
fetched_accessions = {r["accession"] for r in structure_records}

for accession, name in protein_panel.items():
    if accession in fetched_accessions:
        continue
    n_res = rng.integers(100, 600)
    plddt = np.clip(rng.normal(80, 15, n_res), 20, 99)
    coords = np.cumsum(rng.normal(0, 3.8, (n_res, 3)), axis=0)  # rough random-walk backbone
    structure_records.append({
        "accession": accession, "name": name, "n_residues": n_res,
        "mean_pLDDT": np.mean(plddt), "pct_very_high_conf": np.mean(plddt > 90) * 100,
        "pct_low_conf": np.mean(plddt < 50) * 100, "coords": coords, "plddt": plddt,
        "source": "simulated_fallback"
    })

struct_df = pd.DataFrame(structure_records)
print(f"Final structural dataset: {len(struct_df)} proteins ({(struct_df['source']=='real').sum()} real, {(struct_df['source']!='real').sum()} fallback)")
struct_df[["accession", "name", "n_residues", "mean_pLDDT", "source"]]


Final structural dataset: 20 proteins (20 real, 0 fallback)


,accession,name,n_residues,mean_pLDDT,source
0,P00734,Prothrombin,622,83.920177,real
1,P07477,Trypsin-1,247,92.055223,real
2,P00918,Carbonic anhydrase 2,260,97.370615,real
3,P00441,Superoxide dismutase [Cu-Zn],154,97.925065,real
4,P9WHI7,Catalase-peroxidase,587,88.100426,real
5,P00533,EGFR,1210,75.949884,real
6,P08588,Beta-1 adrenergic receptor,477,75.327904,real
7,P41180,Calcium-sensing receptor,1078,75.691438,real
8,P69905,Hemoglobin subunit alpha,142,98.063732,real
9,P02768,Serum albumin,609,92.681281,real


## 4. Fetch Real UniProt Functional Annotations

In [5]:
def fetch_uniprot_function(accession):
    resp = requests.get(f"{UNIPROT_API}/{accession}.json", timeout=30)
    resp.raise_for_status()
    data = resp.json()
    keywords = [kw["name"] for kw in data.get("keywords", [])]
    return keywords

def classify_function(keywords):
    kw_lower = [k.lower() for k in keywords]
    if any(k in kw_lower for k in ["hydrolase", "transferase", "oxidoreductase", "protease", "peroxidase", "lyase", "isomerase"]):
        return "Enzyme"
    if any(k in kw_lower for k in ["receptor", "g-protein coupled receptor"]):
        return "Receptor"
    if any(k in kw_lower for k in ["transport", "ion transport", "iron transport"]):
        return "Transport"
    if any(k in kw_lower for k in ["structural protein", "cytoskeleton", "collagen", "extracellular matrix"]):
        return "Structural"
    if any(k in kw_lower for k in ["immunoglobulin", "adaptive immunity", "innate immunity"]):
        return "Immune"
    if any(k in kw_lower for k in ["hormone"]):
        return "Hormone"
    return "Other"

function_categories = {}
for accession in struct_df["accession"]:
    try:
        keywords = fetch_uniprot_function(accession)
        function_categories[accession] = classify_function(keywords)
    except Exception:
        function_categories[accession] = "Other"

struct_df["function_category"] = struct_df["accession"].map(function_categories)
print(struct_df["function_category"].value_counts())


function_category
Enzyme        6
Transport     4
Other         3
Structural    3
Receptor      2
Immune        1
Hormone       1
Name: count, dtype: int64


## 5. Per-Residue Confidence (pLDDT) Analysis

In [6]:
example_proteins = struct_df.head(4)

fig = go.Figure()
colors = px.colors.qualitative.Set2
for i, (_, row) in enumerate(example_proteins.iterrows()):
    fig.add_trace(go.Scatter(y=row["plddt"], mode="lines", name=f"{row['name']} ({row['accession']})",
                              line=dict(color=colors[i % len(colors)], width=1.5)))

fig.add_hline(y=90, line_dash="dash", line_color="green", annotation_text="Very High Confidence (>90)")
fig.add_hline(y=70, line_dash="dash", line_color="orange", annotation_text="Confident (>70)")
fig.add_hline(y=50, line_dash="dash", line_color="red", annotation_text="Low Confidence (<50)")
fig.update_layout(title="Per-Residue pLDDT Confidence Scores (Example Proteins)",
                   xaxis_title="Residue Position", yaxis_title="pLDDT Score",
                   template="plotly_white", height=500)
fig.show()

print(" pLDDT interpretation: >90 = very high confidence, 70-90 = confident, 50-70 = low confidence,")
print("   <50 = often indicates intrinsically disordered regions")


💡 pLDDT interpretation: >90 = very high confidence, 70-90 = confident, 50-70 = low confidence,
   <50 = often indicates intrinsically disordered regions


In [7]:
fig = px.bar(struct_df.sort_values("mean_pLDDT"), x="mean_pLDDT", y="name", color="function_category",
             orientation='h', title="Mean Prediction Confidence by Protein",
             template="plotly_white", labels={"mean_pLDDT": "Mean pLDDT", "name": "Protein"})
fig.update_layout(height=650)
fig.show()


## 6. Structural Feature Extraction

In [8]:
def radius_of_gyration(coords):
    center = coords.mean(axis=0)
    return np.sqrt(np.mean(np.sum((coords - center) ** 2, axis=1)))

def end_to_end_distance(coords):
    return np.linalg.norm(coords[-1] - coords[0])

struct_df["radius_of_gyration"] = struct_df["coords"].apply(radius_of_gyration)
struct_df["end_to_end_distance"] = struct_df["coords"].apply(end_to_end_distance)
struct_df["compactness"] = struct_df["radius_of_gyration"] / np.sqrt(struct_df["n_residues"])

feature_cols = ["n_residues", "mean_pLDDT", "pct_very_high_conf", "pct_low_conf",
                 "radius_of_gyration", "end_to_end_distance", "compactness"]
struct_df[["name"] + feature_cols].round(2)


,name,n_residues,mean_pLDDT,pct_very_high_conf,pct_low_conf,radius_of_gyration,end_to_end_distance,compactness
0,Prothrombin,622,83.92,55.14,9.81,32.939999,83.220001,1.32
1,Trypsin-1,247,92.06,84.62,4.86,21.330000,98.830002,1.36
2,Carbonic anhydrase 2,260,97.37,98.46,0.38,17.240000,47.959999,1.07
3,Superoxide dismutase [Cu-Zn],154,97.93,98.05,0.00,14.050000,10.940000,1.13
4,Catalase-peroxidase,587,88.10,55.20,0.17,51.619999,23.930000,2.13
5,EGFR,1210,75.95,47.19,22.81,40.130001,105.750000,1.15
6,Beta-1 adrenergic receptor,477,75.33,49.69,26.21,36.869999,46.580002,1.69
7,Calcium-sensing receptor,1078,75.69,42.86,20.50,58.509998,113.760002,1.78
8,Hemoglobin subunit alpha,142,98.06,99.30,0.00,14.470000,19.549999,1.21
9,Serum albumin,609,92.68,88.01,4.11,27.139999,78.139999,1.10


## 7. Exploratory Visualization

In [9]:
fig = px.scatter(struct_df, x="n_residues", y="radius_of_gyration", color="function_category",
                  size="mean_pLDDT", hover_name="name",
                  title="Protein Size vs Compactness (bubble size = mean pLDDT)",
                  template="plotly_white", color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(height=550)
fig.show()

pca_struct = PCA(n_components=2)
struct_pcs = pca_struct.fit_transform(StandardScaler().fit_transform(struct_df[feature_cols]))
struct_df["PC1"], struct_df["PC2"] = struct_pcs[:, 0], struct_pcs[:, 1]

fig2 = px.scatter(struct_df, x="PC1", y="PC2", color="function_category", hover_name="name",
                   title="Structural Feature Space (PCA)", template="plotly_white",
                   color_discrete_sequence=px.colors.qualitative.Bold)
fig2.update_traces(marker=dict(size=11, line=dict(width=0.5, color='white')))
fig2.update_layout(height=550)
fig2.show()


## 8. Function Category Classification (ML)

In [10]:
le = LabelEncoder()
X = struct_df[feature_cols]
y = le.fit_transform(struct_df["function_category"])

func_scaler = StandardScaler()
X_scaled = func_scaler.fit_transform(X)

# Small dataset (20 proteins) — use leave-one-out style evaluation via cross-validation
func_clf = RandomForestClassifier(n_estimators=300, random_state=42)
func_clf.fit(X_scaled, y)  # train on full panel; this model powers the runtime tool below

from sklearn.model_selection import cross_val_predict, LeaveOneOut
loo_preds = cross_val_predict(RandomForestClassifier(n_estimators=300, random_state=42), X_scaled, y, cv=LeaveOneOut())
loo_acc = accuracy_score(y, loo_preds)
print(f"Leave-one-out cross-validation accuracy: {loo_acc:.3f}  (n={len(y)} proteins — small panel, illustrative)")

cm = confusion_matrix(y, loo_preds)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues", x=le.classes_, y=le.classes_,
                 labels=dict(x="Predicted", y="Actual", color="Count"), title="Confusion Matrix (Leave-One-Out CV)")
fig.update_layout(height=500)
fig.show()

importances = pd.Series(func_clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig2 = px.bar(importances, orientation='h', title="Feature Importance for Function Classification",
              template="plotly_white", color=importances.values, color_continuous_scale="Viridis")
fig2.update_layout(height=400, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig2.show()


Leave-one-out cross-validation accuracy: 0.100  (n=20 proteins — small panel, illustrative)


## 9. Interactive 3D Structure Viewer

In [11]:
def show_3d_structure(accession, pdb_text, width=700, height=500):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_text, "pdb")
    view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
    view.zoomTo()
    return view

example_accession = struct_df.iloc[0]["accession"]
if example_accession in pdb_cache:
    print(f"3D structure — {struct_df.iloc[0]['name']} ({example_accession}), colored by pLDDT confidence")
    view = show_3d_structure(example_accession, pdb_cache[example_accession])
    view.show()
else:
    print("Example protein used the simulated fallback (no real PDB available to render) — try the runtime tool below with a real UniProt accession.")


3D structure — Prothrombin (P00734), colored by pLDDT confidence


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 10.  Runtime Prediction — Apna UniProt Accession Daal Kar Structure Analyze Karein

Koi bhi **real human protein ka UniProt accession** (jaise `P04637` for TP53, `P42574` for Caspase-3) daalein — system uska **real AlphaFold structure fetch** karega, confidence analyze karega, **function category predict** karega, aur **interactive 3D structure** dikhayega.


In [12]:
accession_box = widgets.Text(value="P04637", description="UniProt ID:",
                              placeholder="e.g. P04637 (TP53)", style={'description_width': '100px'},
                              layout=widgets.Layout(width='300px'))
predict_btn = widgets.Button(description=" Structure Analyze Karein", button_style='success',
                              layout=widgets.Layout(width='260px', height='38px'))
out = widgets.Output()

def on_predict(b):
    with out:
        clear_output()
        accession = accession_box.value.strip().upper()
        try:
            entry, pdb_text = fetch_alphafold_structure(accession)
            residues, plddt, coords = parse_structure(pdb_text, accession)
        except Exception as e:
            print(f" Could not fetch AlphaFold structure for '{accession}': {str(e)[:100]}")
            print("Tip: sirf real human/model organism UniProt accessions AlphaFold DB mein available hain.")
            return

        n_res = len(plddt)
        mean_plddt = np.mean(plddt)
        pct_vh = np.mean(plddt > 90) * 100
        pct_low = np.mean(plddt < 50) * 100
        rg = radius_of_gyration(coords)
        e2e = end_to_end_distance(coords)
        compact = rg / np.sqrt(n_res)

        feat_row = pd.DataFrame([{
            "n_residues": n_res, "mean_pLDDT": mean_plddt, "pct_very_high_conf": pct_vh,
            "pct_low_conf": pct_low, "radius_of_gyration": rg, "end_to_end_distance": e2e, "compactness": compact
        }])[feature_cols]

        feat_scaled = func_scaler.transform(feat_row)
        pred = func_clf.predict(feat_scaled)[0]
        proba = func_clf.predict_proba(feat_scaled)[0]
        pred_label = le.inverse_transform([pred])[0]

        try:
            uniprot_kw = fetch_uniprot_function(accession)
            protein_name = requests.get(f"{UNIPROT_API}/{accession}.json", timeout=15).json().get("proteinDescription", {}).get("recommendedName", {}).get("fullName", {}).get("value", accession)
        except Exception:
            uniprot_kw, protein_name = [], accession

        top3_idx = proba.argsort()[::-1][:3]
        top3_html = "".join(
            f'<div style="display:flex; justify-content:space-between; font-size:13px; margin-top:4px;">'
            f'<span>{le.classes_[i]}</span><span><b>{proba[i]*100:.1f}%</b></span></div>'
            for i in top3_idx
        )

        html = f"""
        <div style="border:2px solid #2C5364; border-radius:12px; padding:18px; font-family:sans-serif; background:#fafafa;">
            <div style="font-size:19px; font-weight:700; color:#2C5364;"> {protein_name} ({accession})</div>
            <div style="font-size:13px; color:#555; margin-top:6px;">
                {n_res} residues | Mean pLDDT: {mean_plddt:.1f} | Very High Confidence: {pct_vh:.1f}% | Low Confidence: {pct_low:.1f}%
            </div>
            <hr style="margin:12px 0; border:none; border-top:1px solid #ddd;">
            <div style="font-size:16px; font-weight:700; color:#E63946;">Predicted Function: {pred_label}</div>
            <div style="font-size:13px; margin-top:8px;"><b>Top 3 Probabilities:</b></div>
            {top3_html}
        </div>
        """
        display(HTML(html))

        fig = go.Figure()
        fig.add_trace(go.Scatter(y=plddt, mode="lines", line=dict(color="#2E86AB", width=1.5), fill="tozeroy"))
        fig.add_hline(y=90, line_dash="dash", line_color="green")
        fig.add_hline(y=50, line_dash="dash", line_color="red")
        fig.update_layout(title=f"Per-Residue pLDDT — {accession}", xaxis_title="Residue Position",
                           yaxis_title="pLDDT", template="plotly_white", height=350)
        fig.show()

        print("Interactive 3D structure (colored by confidence: blue=high, red=low):")
        view = show_3d_structure(accession, pdb_text)
        view.show()

predict_btn.on_click(on_predict)

display(accession_box)
display(predict_btn)
display(out)


Text(value='P04637', description='UniProt ID:', layout=Layout(width='300px'), placeholder='e.g. P04637 (TP53)'…

Button(button_style='success', description=' Structure Analyze Karein', layout=Layout(height='38px', width='26…

Output()

 example
- `P04637` — TP53 (tumor suppressor)
- `P42574` — Caspase-3 (apoptosis enzyme)
- `P01111` — RAS (signaling protein)
- `P38398` — BRCA1


